# TetheredAI MLB Modeling Lab 09 — Leakage-Safe Features, Correlations, and Champion Export

This notebook is the cleaned-up modeling workflow after discovering that `home_score`, `away_score`, `diff_score`, and `home_margin` leaked the game result into the model.

It is intentionally linear:

1. Load the latest feature parquet.
2. Separate **completed/labeled games** from **unresolved rows** and true **scoring candidates**.
3. Audit for leakage before modeling.
4. Build a clean feature list without postgame outcome fields.
5. Handle missingness and high-missing features.
6. Add batted-ball/statcast interaction terms.
7. Chronologically split train/test.
8. Compare models against a constant home-rate baseline.
9. Diagnose calibration and feature importance.
10. Export a champion model bundle only after the leakage checks pass.

Use log loss, Brier score, calibration, and AUC as the primary selection criteria. Accuracy alone is not enough for betting decisions.

In [ ]:
from pathlib import Path
from datetime import timedelta
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    log_loss,
    brier_score_loss,
    roc_auc_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 250)

# Resolve project root. This works when the notebook is inside notebooks/ or when run from project root.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT / "MLB" / "mlb_betting_app").exists():
    PROJECT_ROOT = PROJECT_ROOT / "MLB" / "mlb_betting_app"

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_PATH = PROCESSED_DIR / "mlb_game_features.parquet"
ODDS_DB_PATH = DATA_DIR / "odds.db"
TARGET_COL = "target_home_win"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FEATURES_PATH:", FEATURES_PATH)
print("ODDS_DB_PATH:", ODDS_DB_PATH)

## 1. Load the feature file and verify the date range

The feature file is the source of truth. This cell shows exactly what date range and target coverage you are modeling from.

In [ ]:
assert FEATURES_PATH.exists(), f"Missing feature file: {FEATURES_PATH}"

features = pd.read_parquet(FEATURES_PATH)
features = features.copy()

if "official_date" in features.columns:
    features["official_date"] = pd.to_datetime(features["official_date"], errors="coerce")
if "game_datetime_utc" in features.columns:
    features["game_datetime_utc"] = pd.to_datetime(features["game_datetime_utc"], utc=True, errors="coerce")

print("Rows:", len(features))
print("Columns:", len(features.columns))
print("All official_date range:", features["official_date"].min(), "to", features["official_date"].max())

completed_all = features[features[TARGET_COL].notna()].copy()
completed_all[TARGET_COL] = completed_all[TARGET_COL].astype(int)
unresolved_all = features[features[TARGET_COL].isna()].copy()

print("Completed/labeled rows:", len(completed_all))
print("Unresolved rows:", len(unresolved_all))
print("Completed date range:", completed_all["official_date"].min(), "to", completed_all["official_date"].max())
if len(unresolved_all):
    print("Unresolved date range:", unresolved_all["official_date"].min(), "to", unresolved_all["official_date"].max())

bb_cols_preview = [
    c for c in features.columns
    if "batted_ball" in c.lower() or "distance" in c.lower() or "avg_ev" in c.lower()
]
print("Batted-ball/distance columns:", len(bb_cols_preview))
display(pd.Series(bb_cols_preview, name="batted_ball_distance_cols").head(100))

print("Target distribution:")
display(completed_all[TARGET_COL].value_counts(dropna=False).to_frame("count"))

## 2. Distinguish unresolved rows from true upcoming scoring candidates

`target_home_win.isna()` only means **unresolved/unlabeled**. It does not necessarily mean upcoming. Historical postponed/canceled rows and far-future schedule rows can also be unresolved.

In [ ]:
AS_OF_DATE = pd.Timestamp.today(tz="America/Chicago").date()
DAYS_FORWARD = 14

features["official_date_dt"] = pd.to_datetime(features["official_date"], errors="coerce").dt.date
unresolved = features[features[TARGET_COL].isna()].copy()

start_date = AS_OF_DATE
end_date = AS_OF_DATE + timedelta(days=DAYS_FORWARD)

date_mask = (
    (unresolved["official_date_dt"] >= start_date) &
    (unresolved["official_date_dt"] <= end_date)
)

status_mask = pd.Series(True, index=unresolved.index)
if "abstract_state" in unresolved.columns:
    status_mask &= unresolved["abstract_state"].astype(str).str.lower().isin(["preview", "live"])
if "detailed_state" in unresolved.columns:
    bad_states = "postponed|cancelled|canceled|suspended|final|completed"
    status_mask &= ~unresolved["detailed_state"].astype(str).str.lower().str.contains(bad_states, na=False)

scoring_candidates = unresolved[date_mask & status_mask].copy()
historical_unresolved = unresolved[unresolved["official_date_dt"] < start_date].copy()
far_future_unresolved = unresolved[unresolved["official_date_dt"] > end_date].copy()

print("AS_OF_DATE:", AS_OF_DATE)
print("Scoring window:", start_date, "to", end_date)
print("Unresolved rows:", len(unresolved))
print("Scoring candidates:", len(scoring_candidates))
print("Historical unresolved:", len(historical_unresolved))
print("Far-future unresolved:", len(far_future_unresolved))

if len(scoring_candidates):
    print("Scoring candidate range:", scoring_candidates["official_date"].min(), "to", scoring_candidates["official_date"].max())
    display(
        scoring_candidates[
            [c for c in [
                "game_pk", "official_date", "game_datetime_utc", "away_team_name", "home_team_name",
                "abstract_state", "detailed_state"
            ] if c in scoring_candidates.columns]
        ].sort_values(["official_date", "game_datetime_utc"]).head(50)
    )
else:
    print("No scoring candidates in the current date window.")

print("Historical unresolved sample:")
display(
    historical_unresolved[
        [c for c in [
            "game_pk", "official_date", "game_datetime_utc", "away_team_name", "home_team_name",
            "abstract_state", "detailed_state"
        ] if c in historical_unresolved.columns]
    ].sort_values(["official_date", "game_datetime_utc"]).head(20)
)

print("Far-future unresolved sample:")
display(
    far_future_unresolved[
        [c for c in [
            "game_pk", "official_date", "game_datetime_utc", "away_team_name", "home_team_name",
            "abstract_state", "detailed_state"
        ] if c in far_future_unresolved.columns]
    ].sort_values(["official_date", "game_datetime_utc"]).head(20)
)

## 3. Choose modeling window

The full dataset may include 2022 because the GCP backfill used `DAYS_BACK=1400`. That is okay. Use `MIN_TRAIN_DATE` to control the modeling window.

Set `MIN_TRAIN_DATE = None` to include all labeled rows.

In [ ]:
MIN_TRAIN_DATE = "2023-01-01"  # Change to None to include all labeled rows, including 2022.
MAX_TRAIN_DATE = None           # Usually leave as None.

completed = completed_all.copy()

if MIN_TRAIN_DATE is not None:
    completed = completed[completed["official_date"] >= pd.Timestamp(MIN_TRAIN_DATE)].copy()
if MAX_TRAIN_DATE is not None:
    completed = completed[completed["official_date"] <= pd.Timestamp(MAX_TRAIN_DATE)].copy()

completed[TARGET_COL] = completed[TARGET_COL].astype(int)

print("Modeling rows:", len(completed))
print("Modeling date range:", completed["official_date"].min(), "to", completed["official_date"].max())
print("Home win rate:", completed[TARGET_COL].mean())
display(completed[TARGET_COL].value_counts(normalize=True).rename("rate").to_frame())

## 4. Build candidate feature columns

This first pass identifies numeric columns that could be features. Leakage filtering happens in the next section.

In [ ]:
METADATA_COLS = {
    TARGET_COL,
    "game_pk",
    "run_id",
    "scored_at_utc",
    "official_date",
    "official_date_dt",
    "game_datetime_utc",
    "home_team_name",
    "away_team_name",
    "home_team_id",
    "away_team_id",
    "venue_name",
    "abstract_state",
    "detailed_state",
}

# Keep market columns separate. You can test them later, but start with pure-baseball models.
MARKET_TOKENS = [
    "market_", "moneyline", "spread", "total_points", "over_price", "under_price", "book_count", "vig", "implied_prob"
]

def is_market_col(col: str) -> bool:
    low = col.lower()
    return any(tok in low for tok in MARKET_TOKENS)

candidate_feature_cols = [
    c for c in completed.columns
    if c not in METADATA_COLS
    and pd.api.types.is_numeric_dtype(completed[c])
]

market_cols = [c for c in candidate_feature_cols if is_market_col(c)]
statcast_cols = [c for c in candidate_feature_cols if "statcast" in c.lower() or "sc_" in c.lower()]
batted_ball_cols = [
    c for c in candidate_feature_cols
    if "batted_ball" in c.lower() or "distance" in c.lower() or "avg_ev" in c.lower()
]

print("Candidate numeric features:", len(candidate_feature_cols))
print("Market columns:", len(market_cols))
print("Statcast columns:", len(statcast_cols))
print("Batted-ball/distance columns:", len(batted_ball_cols))

## 5. Leakage audit and clean feature list

The known leaking columns are:

- `home_score`
- `away_score`
- `diff_score`
- `home_margin`
- `target_home_win`

They define or directly reveal the result. They should be used only for labels/evaluation, never as model features.

In [ ]:
LEAKY_COLS_EXACT = {
    "home_score",
    "away_score",
    "diff_score",
    "home_margin",
    "target_home_win",
    # Extra exact names sometimes used in sports datasets.
    "home_runs",
    "away_runs",
    "total_runs",
    "run_total",
    "margin",
}

ROLLING_TOKENS = [
    "_last3", "_last5", "_last10", "_last20", "_last30", "_last40", "_season_to_date", "_pre"
]

HARD_LEAK_PATTERNS = [
    "winner", "winning", "losing", "final", "result", "outcome", "actual", "post_"
]


def is_probably_leaky_feature(col: str) -> bool:
    c = col.lower()

    if c in LEAKY_COLS_EXACT:
        return True

    if any(p in c for p in HARD_LEAK_PATTERNS):
        return True

    # Raw same-game boxscore columns are suspicious unless they are explicitly rolling/pregame.
    # Example to drop: home_team_box_runs
    # Example to keep: home_team_box_runs_last20 or home_team_box_runs_season_to_date
    if c.startswith(("home_team_box_", "away_team_box_", "diff_team_box_")):
        if not any(tok in c for tok in ROLLING_TOKENS):
            return True

    # Raw identifiers should not be model inputs.
    if c.endswith("_id") or c in {"game_pk"}:
        return True

    return False

hard_suspects = [c for c in candidate_feature_cols if c.lower() in LEAKY_COLS_EXACT or any(p in c.lower() for p in HARD_LEAK_PATTERNS)]
same_game_box_suspects = [
    c for c in candidate_feature_cols
    if c.lower().startswith(("home_team_box_", "away_team_box_", "diff_team_box_"))
    and not any(tok in c.lower() for tok in ROLLING_TOKENS)
]

print("Hard leakage suspects:", len(hard_suspects))
display(pd.Series(hard_suspects, name="hard_leakage_suspects").head(200))

print("Potential same-game boxscore suspects:", len(same_game_box_suspects))
display(pd.Series(same_game_box_suspects, name="same_game_boxscore_suspects").head(250))

leakage_removed_cols = sorted([c for c in candidate_feature_cols if is_probably_leaky_feature(c)])
feature_cols_no_leak = [c for c in candidate_feature_cols if c not in leakage_removed_cols]

print("Candidate features:", len(candidate_feature_cols))
print("Removed by leakage filter:", len(leakage_removed_cols))
print("After leakage filter:", len(feature_cols_no_leak))
display(pd.Series(leakage_removed_cols, name="removed_by_leakage_filter").head(300))

### Single-feature leakage test

This identifies features that predict the target suspiciously well by themselves. Anything above ~0.90 single-feature AUC deserves inspection. The known leaking score columns should disappear after cleaning.

In [ ]:
def single_feature_auc_table(df: pd.DataFrame, cols: list[str], target_col: str = TARGET_COL) -> pd.DataFrame:
    y = df[target_col].astype(int)
    rows = []
    for c in cols:
        if c not in df.columns or not pd.api.types.is_numeric_dtype(df[c]):
            continue
        tmp = pd.DataFrame({"x": df[c], "y": y}).dropna()
        if tmp["x"].nunique() < 2 or tmp["y"].nunique() < 2:
            continue
        try:
            auc = roc_auc_score(tmp["y"], tmp["x"])
            auc = max(auc, 1 - auc)
            rows.append((c, auc, len(tmp), tmp["x"].nunique()))
        except Exception:
            pass
    return pd.DataFrame(rows, columns=["feature", "single_feature_auc", "n", "n_unique"]).sort_values(
        "single_feature_auc", ascending=False
    )

print("Top single-feature AUC BEFORE leakage cleaning:")
single_auc_before = single_feature_auc_table(completed, candidate_feature_cols)
display(single_auc_before.head(100))

print("Top single-feature AUC AFTER leakage cleaning:")
single_auc_after = single_feature_auc_table(completed, feature_cols_no_leak)
display(single_auc_after.head(100))

suspicious_after = single_auc_after[single_auc_after["single_feature_auc"] >= 0.90]
if len(suspicious_after):
    print("WARNING: suspicious high-AUC features remain after leakage cleaning.")
    display(suspicious_after)
else:
    print("No single feature above 0.90 AUC after leakage cleaning.")

## 6. Missingness handling

Rules:

- Drop very-high-missing features above `DROP_IF_MISSING_GT`.
- Drop stolen-base success-rate features because missingness is denominator-driven and unstable.
- Keep moderate-missing starter Statcast features; the model pipeline uses median imputation plus missing indicators.

In [ ]:
DROP_IF_MISSING_GT = 0.35
MANUAL_DROP_CONTAINS = ["sb_success_rate"]

missing_pct = completed[feature_cols_no_leak].isna().mean().sort_values(ascending=False)
missing_summary = missing_pct.to_frame("missing_pct")
print("Top missingness after leakage filtering:")
display(missing_summary.head(100))

drop_high_missing = missing_pct[missing_pct > DROP_IF_MISSING_GT].index.tolist()
manual_drop = [c for c in feature_cols_no_leak if any(token in c.lower() for token in MANUAL_DROP_CONTAINS)]

missing_removed_cols = sorted(set(drop_high_missing + manual_drop))
feature_cols_clean = [c for c in feature_cols_no_leak if c not in missing_removed_cols]

print("Removed for missingness/manual rules:", len(missing_removed_cols))
display(pd.Series(missing_removed_cols, name="removed_missing_or_manual").head(300))
print("Clean feature count:", len(feature_cols_clean))

## 7. Correlation diagnostics on clean features

Correlation is a diagnostic, not a final model selector. It helps find redundant signals, broken features, and interaction candidates.

In [ ]:
numeric_clean = completed[feature_cols_clean].select_dtypes(include="number").copy()

corr_to_target = (
    numeric_clean.assign(target_home_win=completed[TARGET_COL].values)
    .corr(numeric_only=True)[TARGET_COL]
    .drop(TARGET_COL)
    .dropna()
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

print("Top absolute correlations to target:")
display(corr_to_target.head(80).to_frame("corr_to_home_win"))

print("Most positive correlations:")
display(corr_to_target.sort_values(ascending=False).head(50).to_frame("corr_to_home_win"))

print("Most negative correlations:")
display(corr_to_target.sort_values(ascending=True).head(50).to_frame("corr_to_home_win"))

In [ ]:
TOP_N_FOR_REDUNDANCY = 120
corr_features = corr_to_target.abs().head(TOP_N_FOR_REDUNDANCY).index.tolist()

if len(corr_features) >= 2:
    corr_abs = numeric_clean[corr_features].corr(numeric_only=True).abs()
    upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))
    high_corr_pairs = (
        upper.stack()
        .reset_index()
        .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "abs_corr"})
        .sort_values("abs_corr", ascending=False)
    )
    display(high_corr_pairs.head(120))
else:
    print("Not enough features for redundancy table.")

In [ ]:
heatmap_features = corr_to_target.abs().head(30).index.tolist()

if len(heatmap_features) >= 2:
    mat = numeric_clean[heatmap_features].corr(numeric_only=True)
    plt.figure(figsize=(12, 10))
    plt.imshow(mat, aspect="auto")
    plt.xticks(range(len(heatmap_features)), heatmap_features, rotation=90)
    plt.yticks(range(len(heatmap_features)), heatmap_features)
    plt.colorbar()
    plt.title("Correlation among top target-correlated clean features")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough heatmap features.")

## 8. Add baseball-logical interaction terms

These are deliberately limited. Promote interactions only if they improve holdout log loss/Brier and calibration.

In [ ]:
def add_interaction_terms(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    out = df.copy()
    created = []

    specs = [
        # Team contact quality × starter contact allowed.
        ("diff_team_off_sc_avg_batted_ball_ev_last20", "diff_starter_statcast_sc_avg_batted_ball_ev_allowed_last10", "int_team_ev_x_starter_ev_allowed"),
        ("diff_team_off_sc_avg_batted_ball_distance_last20", "diff_starter_statcast_sc_avg_batted_ball_distance_allowed_last10", "int_team_dist_x_starter_dist_allowed"),
        ("diff_team_off_sc_barrel_rate_last20", "diff_starter_statcast_sc_barrel_rate_allowed_last10", "int_team_barrel_x_starter_barrel_allowed"),
        ("diff_team_off_sc_hard_hit_rate_last20", "diff_starter_statcast_sc_hard_hit_rate_allowed_last10", "int_team_hardhit_x_starter_hardhit_allowed"),

        # Platoon offense × starter skill.
        ("diff_team_vs_hand_ops_last20", "diff_starter_k_minus_bb_per_bf_last10", "int_vs_hand_ops_x_starter_kbb"),
        ("diff_team_vs_hand_sc_sc_woba_last20", "diff_starter_statcast_sc_woba_allowed_last10", "int_vs_hand_woba_x_starter_woba_allowed"),

        # Bullpen quality/fatigue × opponent contact.
        ("diff_bullpen_sc_woba_allowed_last20", "diff_team_off_sc_hard_hit_rate_last20", "int_bullpen_woba_x_team_hardhit"),
        ("diff_sc_bullpen_pitches_last3", "diff_team_off_sc_barrel_rate_last20", "int_bullpen_work_x_team_barrel"),

        # ELO/team strength × starter edge.
        ("diff_elo_pre", "diff_starter_statcast_sc_k_rate_last20", "int_elo_x_starter_k_rate"),
        ("elo_home_win_prob", "diff_starter_statcast_sc_woba_allowed_last20", "int_elo_prob_x_starter_woba_allowed"),
    ]

    for a, b, new_col in specs:
        if a in out.columns and b in out.columns:
            out[new_col] = out[a] * out[b]
            created.append(new_col)

    return out, created

features_i, interaction_cols = add_interaction_terms(features)
completed_i = features_i.loc[completed.index].copy()

# Only keep interactions that are numeric and non-leaky by construction.
interaction_cols = [c for c in interaction_cols if c in completed_i.columns and pd.api.types.is_numeric_dtype(completed_i[c])]
feature_cols_with_interactions = feature_cols_clean + interaction_cols

print("Created interaction columns:", len(interaction_cols))
display(pd.Series(interaction_cols, name="interaction_cols"))
print("Feature count with interactions:", len(feature_cols_with_interactions))

## 9. Chronological train/test split and baseline

Do not drop columns from the split dataframe. Keep the target available, and control model inputs through `clean_feature_cols` / `feature_cols_with_interactions`.

In [ ]:
def chronological_split(df: pd.DataFrame, test_frac: float = 0.20):
    sort_cols = [c for c in ["official_date", "game_datetime_utc", "game_pk"] if c in df.columns]
    df = df.sort_values(sort_cols).reset_index(drop=True)
    split_idx = int(len(df) * (1 - test_frac))
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

train_df, test_df = chronological_split(completed_i, test_frac=0.20)

print("Train rows:", len(train_df), train_df["official_date"].min(), "to", train_df["official_date"].max())
print("Test rows:", len(test_df), test_df["official_date"].min(), "to", test_df["official_date"].max())

y_train = train_df[TARGET_COL].astype(int)
y_test = test_df[TARGET_COL].astype(int)

baseline_train_rate = y_train.mean()
baseline_probs = np.repeat(baseline_train_rate, len(y_test))

baseline_metrics = {
    "model_name": "constant_train_home_rate",
    "feature_count": 0,
    "n_test": len(y_test),
    "avg_pred": float(baseline_probs.mean()),
    "actual_rate": float(y_test.mean()),
    "log_loss": float(log_loss(y_test, baseline_probs)),
    "brier": float(brier_score_loss(y_test, baseline_probs)),
    "roc_auc": 0.5,
    "accuracy_50pct": float(accuracy_score(y_test, baseline_probs >= 0.5)),
    "suspicious_metric": False,
}

display(pd.DataFrame([baseline_metrics]))

## 10. Model feature sets

`baseline_no_market` now means **cleaned, non-market, non-leaky** features. It should not contain score/margin/result columns.

In [ ]:
def cols_matching(cols, include_any=(), exclude_any=()):
    out = []
    for c in cols:
        low = c.lower()
        if include_any and not any(x in low for x in include_any):
            continue
        if exclude_any and any(x in low for x in exclude_any):
            continue
        out.append(c)
    return out

pure_base_cols = [c for c in feature_cols_clean if c not in market_cols]
pure_base_cols_i = [c for c in feature_cols_with_interactions if c not in market_cols]

feature_sets = {
    "baseline_no_market_clean": pure_base_cols,
    "enhanced_with_interactions_clean": pure_base_cols_i,
    "batted_ball_only": cols_matching(
        pure_base_cols_i,
        include_any=("batted_ball", "avg_ev", "max_ev", "p90_batted", "distance", "hard_hit", "barrel")
    ),
    "statcast_only": [c for c in pure_base_cols_i if (c in statcast_cols or c in interaction_cols)],
    "starter_statcast": cols_matching(pure_base_cols_i, include_any=("starter_statcast",)),
    "team_statcast": cols_matching(pure_base_cols_i, include_any=("team_off_sc", "team_vs_hand_sc")),
    "starter_plus_elo": sorted(set(cols_matching(pure_base_cols_i, include_any=("starter",)) + cols_matching(pure_base_cols_i, include_any=("elo",)))),
}

# Remove empty or tiny feature sets.
feature_sets = {k: v for k, v in feature_sets.items() if len(v) >= 5}

for name, cols in feature_sets.items():
    bad = [c for c in cols if c not in train_df.columns]
    assert not bad, f"{name} has missing columns: {bad[:10]}"
    leaked = [c for c in cols if is_probably_leaky_feature(c)]
    assert not leaked, f"{name} has leaky columns: {leaked[:20]}"

summary = pd.DataFrame([{"feature_set": k, "feature_count": len(v)} for k, v in feature_sets.items()])
display(summary)

## 11. Define models

SVM is included as optional because it can be slow with thousands of features. Set `RUN_SVM = True` if you want to test it.

In [ ]:
def make_pipeline(model, scale=False):
    steps = [("imputer", SimpleImputer(strategy="median", add_indicator=True))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", model))
    return Pipeline(steps)

models = {
    "logit_l2": make_pipeline(LogisticRegression(max_iter=5000, C=0.5, solver="lbfgs"), scale=True),
    "random_forest": make_pipeline(RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=25,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    )),
    "extra_trees": make_pipeline(ExtraTreesClassifier(
        n_estimators=500,
        min_samples_leaf=25,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    )),
    "hist_gbdt": make_pipeline(HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.03,
        max_leaf_nodes=15,
        l2_regularization=0.1,
        random_state=42,
    )),
}

try:
    from xgboost import XGBClassifier
    models["xgboost"] = make_pipeline(XGBClassifier(
        n_estimators=300,
        max_depth=2,
        learning_rate=0.03,
        subsample=0.9,
        colsample_bytree=0.8,
        min_child_weight=10,
        reg_alpha=0.0,
        reg_lambda=2.0,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    ))
except Exception as e:
    print("XGBoost unavailable:", e)

try:
    from lightgbm import LGBMClassifier
    models["lightgbm"] = make_pipeline(LGBMClassifier(
        n_estimators=400,
        max_depth=3,
        learning_rate=0.02,
        subsample=0.9,
        colsample_bytree=0.8,
        min_child_samples=50,
        reg_alpha=0.0,
        reg_lambda=2.0,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    ))
except Exception as e:
    print("LightGBM unavailable:", e)

RUN_SVM = False
if RUN_SVM:
    from sklearn.svm import SVC
    models["svm_rbf"] = make_pipeline(SVC(C=1.0, kernel="rbf", probability=True, random_state=42), scale=True)

print("Models:", list(models.keys()))

## 12. Fit and evaluate model grid

The suspicious-metric flag catches impossible-looking results. If you see AUC/accuracy near 1.0 or log loss near zero, stop and rerun leakage audit.

In [ ]:
def evaluate_model(name, model, cols):
    X_train = train_df[cols].copy()
    X_test = test_df[cols].copy()

    fitted = clone(model)
    fitted.fit(X_train, y_train)

    if hasattr(fitted, "predict_proba"):
        p = fitted.predict_proba(X_test)[:, 1]
    else:
        raw = fitted.decision_function(X_test)
        p = 1 / (1 + np.exp(-raw))

    p = np.clip(p, 1e-6, 1 - 1e-6)
    pred = (p >= 0.5).astype(int)

    metrics = {
        "model_name": name,
        "feature_count": len(cols),
        "n_test": len(y_test),
        "avg_pred": float(np.mean(p)),
        "actual_rate": float(y_test.mean()),
        "log_loss": float(log_loss(y_test, p)),
        "brier": float(brier_score_loss(y_test, p)),
        "roc_auc": float(roc_auc_score(y_test, p)),
        "accuracy_50pct": float(accuracy_score(y_test, pred)),
    }

    metrics["suspicious_metric"] = bool(
        metrics["roc_auc"] >= 0.95 or
        metrics["accuracy_50pct"] >= 0.90 or
        metrics["log_loss"] <= 0.25
    )
    return fitted, p, metrics

results = [baseline_metrics]
fitted_models = {}
preds_by_name = {}

for fs_name, cols in feature_sets.items():
    if len(cols) == 0:
        continue

    for model_name, model in models.items():
        full_name = f"{fs_name}__{model_name}"
        try:
            fitted, p, metrics = evaluate_model(full_name, model, cols)
            results.append(metrics)
            fitted_models[full_name] = (fitted, cols)
            preds_by_name[full_name] = p
            print("finished", full_name, metrics)
        except Exception as e:
            print("FAILED", full_name, repr(e))

results_df = pd.DataFrame(results).sort_values(["suspicious_metric", "log_loss", "brier"]).reset_index(drop=True)
display(results_df)

if results_df["suspicious_metric"].any():
    print("WARNING: At least one model produced suspiciously strong metrics. Do not export those models.")
    display(results_df[results_df["suspicious_metric"]])

## 13. Champion diagnostics

This selects the best non-suspicious trained model by log loss. The constant baseline is not exportable.

In [ ]:
eligible = results_df[
    (~results_df["suspicious_metric"]) &
    (results_df["model_name"] != "constant_train_home_rate") &
    (results_df["model_name"].isin(fitted_models.keys()))
].copy()

assert len(eligible), "No eligible non-suspicious trained models found. Recheck leakage and feature filtering."

best_name = eligible.sort_values(["log_loss", "brier"]).iloc[0]["model_name"]
best_model, best_cols = fitted_models[best_name]
best_p = preds_by_name[best_name]

print("Best eligible model:", best_name)
print("Feature count:", len(best_cols))
display(results_df[results_df["model_name"].eq(best_name)])

print("\nClassification report at 50% threshold:")
print(classification_report(y_test, (best_p >= 0.5).astype(int), digits=3))

print("Confusion matrix:")
display(pd.DataFrame(
    confusion_matrix(y_test, (best_p >= 0.5).astype(int)),
    index=["actual_away", "actual_home"],
    columns=["pred_away", "pred_home"],
))

calib = pd.DataFrame({"p": best_p, "y": y_test.values})
calib["bucket"] = pd.cut(calib["p"], bins=[0, .35, .40, .45, .50, .55, .60, .65, .70, 1.0], include_lowest=True)
calib_table = calib.groupby("bucket", observed=False).agg(
    games=("y", "size"),
    avg_pred_prob=("p", "mean"),
    actual_home_win_rate=("y", "mean"),
)
calib_table["calibration_error"] = calib_table["actual_home_win_rate"] - calib_table["avg_pred_prob"]
display(calib_table)

plt.figure(figsize=(7, 5))
plt.plot([0, 1], [0, 1], linestyle="--")
plt.scatter(calib_table["avg_pred_prob"], calib_table["actual_home_win_rate"])
plt.xlabel("Average predicted home-win probability")
plt.ylabel("Actual home-win rate")
plt.title(f"Calibration: {best_name}")
plt.tight_layout()
plt.show()

## 14. Permutation importance

This can be slow. It uses the holdout period only and scores by negative log loss.

In [ ]:
RUN_PERMUTATION_IMPORTANCE = False

if RUN_PERMUTATION_IMPORTANCE:
    X_test_best = test_df[best_cols].copy()
    perm = permutation_importance(
        best_model,
        X_test_best,
        y_test,
        scoring="neg_log_loss",
        n_repeats=5,
        random_state=42,
        n_jobs=-1,
    )
    imp = pd.DataFrame({
        "feature": best_cols,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    }).sort_values("importance_mean", ascending=False)
    display(imp.head(100))
else:
    print("Permutation importance skipped. Set RUN_PERMUTATION_IMPORTANCE = True to run it.")

## 15. Optional champion export

Set `APPROVE_EXPORT = True` only after:

- Leakage audit shows no impossible features.
- Champion metrics are not suspicious.
- Calibration looks reasonable.
- The champion beats the constant home-rate baseline on log loss and Brier.

The exported bundle contains the fitted model and the exact feature columns required by production scoring.

In [ ]:
APPROVE_EXPORT = False

best_metrics = results_df[results_df["model_name"].eq(best_name)].iloc[0].to_dict()
constant_metrics = results_df[results_df["model_name"].eq("constant_train_home_rate")].iloc[0].to_dict()

beats_baseline = (
    best_metrics["log_loss"] < constant_metrics["log_loss"] and
    best_metrics["brier"] < constant_metrics["brier"]
)

print("Best model:", best_name)
print("Beats constant baseline on log loss and Brier:", beats_baseline)
print("Best metrics:", best_metrics)
print("Constant baseline metrics:", constant_metrics)

if APPROVE_EXPORT:
    assert not best_metrics["suspicious_metric"], "Refusing export: champion metrics are suspicious."
    assert beats_baseline, "Refusing export: champion does not beat constant baseline on log loss and Brier."

    import joblib

    bundle = {
        "model": best_model,
        "feature_cols": best_cols,
        "model_name": best_name,
        "metrics": best_metrics,
        "target_col": TARGET_COL,
        "min_train_date": MIN_TRAIN_DATE,
        "max_train_date": MAX_TRAIN_DATE,
        "train_start": str(train_df["official_date"].min()),
        "train_end": str(train_df["official_date"].max()),
        "test_start": str(test_df["official_date"].min()),
        "test_end": str(test_df["official_date"].max()),
        "created_from_features_path": str(FEATURES_PATH),
        "leakage_removed_cols": leakage_removed_cols,
        "missing_removed_cols": missing_removed_cols,
    }

    model_path = MODEL_DIR / "mlb_moneyline_champion.joblib"
    metadata_path = MODEL_DIR / "mlb_moneyline_champion_metadata.json"

    joblib.dump(bundle, model_path)

    metadata = {k: v for k, v in bundle.items() if k != "model"}
    metadata_path.write_text(json.dumps(metadata, default=str, indent=2))

    print("Saved:", model_path)
    print("Saved:", metadata_path)
else:
    print("APPROVE_EXPORT is False; not exporting.")

## 16. Upload champion model to GCS after export

After setting `APPROVE_EXPORT = True` and rerunning the export cell, upload the files from Cloud Shell or your local terminal:

```bash
export PROJECT_ID="tetheredai-preds"
export BUCKET="tetheredai-mlb-state-${PROJECT_ID}"

cd ~/TetheredAI/MLB/mlb_betting_app

gcloud storage cp models/mlb_moneyline_champion.joblib   "gs://${BUCKET}/mlb/models/mlb_moneyline_champion.joblib"

gcloud storage cp models/mlb_moneyline_champion_metadata.json   "gs://${BUCKET}/mlb/models/mlb_moneyline_champion_metadata.json"
```

Then run the daily scoring Cloud Run job.